# T-REX 음식 모델 — AI Hub 한식 + Food-101 양식·일식 35종 머지 학습

**런타임 → 모두 실행** 한 번이면 끝까지 돈다. 결과는 Drive `MyDrive/trex/merge_out/` 에 남는다(자는 동안 창을 닫아도 된다 — Pro+ 백그라운드 실행).

- **왜 머지인가**: 학습된 모델에 새 음식만 이어서 학습하면 기존 음식을 잊는다(파국적 망각). 옛 데이터(AI Hub)와 새 데이터(Food-101)를 **한 데이터셋으로 합쳐** 다시 학습한다.
- **새 음식 35종**: 피자·햄버거·핫도그·라멘·타코야키·스테이크·타코·나초·시저샐러드·클럽샌드위치·그릴드치즈샌드위치·프렌치토스트·팬케이크·와플·도넛·아이스크림·치즈케이크·초콜릿케이크·컵케이크·마카롱·티라미수·추로스·어니언링·마늘빵·라자냐·리조또·맥앤치즈·피시앤칩스·팟타이·에그베네딕트·폭립·퀘사디아·오믈렛·에다마메·스프링롤. AI Hub 에 이미 있는 음식(초밥·만두·볶음밥·감자튀김 등)은 뺐다.
- **Food-101 에는 박스가 없다** → 위치 모델(YOLOE)로 음식 자리를, 기존 모델(best.pt)로 옆의 한식(감자튀김 등)을 가짜 라벨로 만든다.
- **확인하는 것**: 새 모델이 **기존 음식 성능을 잃지 않았는지**(이전 모델과 같은 AI Hub 검증 사진으로 비교)와 새 35종 성능.

### 필요한 것
- Drive `MyDrive/trex/dataset.tar`(AI Hub, 이전 학습에 쓴 것). 없으면 `dataset_342.tar` 를 찾는다.
- 있으면 좋은 것: `MyDrive/trex/runs/food/weights/best.pt`(이전 학습 결과) — 이어서 학습하고, 한식 가짜 라벨·망각 비교에 쓴다.
- 런타임: **G4 → A100 → L4** 중 잡히는 GPU + 고용량 RAM. **런타임 설정에서 '백그라운드 실행' 을 켠다.**

### 예상 시간(G4 기준, 대략)
압축 풀기 5분 · Food-101 받기 5~20분 · 가짜 라벨 10~15분 · 학습 1.5~3시간 · 변환 10분.


In [ ]:
# 1. 환경 + GPU 확인
!pip install -q -U ultralytics
import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'GPU 가 없다 — 런타임 → 런타임 유형 변경에서 GPU 를 고른다'
p = torch.cuda.get_device_properties(0); print(f'GPU: {p.name}, VRAM {p.total_memory / 1e9:.0f} GB')
!free -g | head -2; df -h /content | tail -1


In [ ]:
# 2. Drive 연결 + 경로
from google.colab import drive
drive.mount('/content/drive')
import os, glob
from pathlib import Path
TREX = Path('/content/drive/MyDrive/trex')
cands = [TREX / 'dataset.tar', TREX / 'dataset_342.tar']
AIHUB_TAR = next((c for c in cands if c.exists()), None)
assert AIHUB_TAR, f'AI Hub 데이터셋이 없다: {cands}'
BASE_PT = TREX / 'runs_merge2/merge/weights/best.pt'
BASE_PT = str(BASE_PT) if BASE_PT.exists() else None
RUNS_DIR = str(TREX / 'runs_merge3')          # 학습 가중치(끊겨도 이어서 학습)
OUT_DIR = TREX / 'merge_out3'; OUT_DIR.mkdir(parents=True, exist_ok=True)
print('AI Hub:', AIHUB_TAR, '| 이전 모델:', BASE_PT or '없음 — yolov8n 에서 시작, 한식 가짜 라벨·망각 비교 생략')


In [ ]:
# 3. AI Hub 데이터셋 풀기
import shutil
DATASET = Path('/content/dataset')
if not (DATASET / 'data.yaml').exists():
    !rm -rf /content/dataset && mkdir -p /content/dataset
    !tar -xf "{AIHUB_TAR}" -C /content/dataset
    if not (DATASET / 'data.yaml').exists():   # tar 안에 폴더가 한 겹 더 있으면 끌어올린다
        inner = [d for d in DATASET.iterdir() if d.is_dir() and (d / 'data.yaml').exists()]
        assert len(inner) == 1, os.listdir(DATASET)
        for item in inner[0].iterdir(): shutil.move(str(item), str(DATASET / item.name))
        inner[0].rmdir()
OLD_NAMES = [l.strip() for l in (DATASET / 'food_labels.txt').read_text(encoding='utf-8').splitlines() if l.strip()]
# 머지 전 AI Hub 검증 사진 목록(망각 비교용) — 머지 후에는 Food-101 사진이 섞인다
AIHUB_VAL = sorted(str(p) for p in (DATASET / 'images' / 'val').glob('*.jpg') if not p.name.startswith('f101_'))
print(f'AI Hub {len(OLD_NAMES)}종 · train {len(list((DATASET / "images" / "train").glob("*.jpg")))} · val {len(AIHUB_VAL)}')


In [ ]:
# 4. Food-101 받기 (ETH 서버, 5GB) — 끊기면 이어받는다
F101 = Path('/content/food-101')
if not (F101 / 'meta' / 'train.txt').exists():
    !for i in 1 2 3 4 5 6; do curl -sL -C - --retry 3 -o /content/food-101.tar.gz https://data.vision.ee.ethz.ch/cvl/food-101.tar.gz && break; sleep 10; done
    !ls -la /content/food-101.tar.gz && tar -xzf /content/food-101.tar.gz -C /content && rm /content/food-101.tar.gz
assert (F101 / 'meta' / 'train.txt').exists(), 'Food-101 을 받지 못했다'
print('Food-101 준비됨:', len(os.listdir(F101 / 'images')), '종')


In [ ]:
%%writefile /content/merge_food101.py
# AI Hub 한식 데이터셋(YOLO 형식)에 Food-101 의 양식·일식 35종을 합친다 — 새 음식만 따로 학습하면 기존 음식을 잊어버리므로
# (파국적 망각) 옛 데이터와 새 데이터를 한 데이터셋으로 머지해 다시 학습한다. Colab 노트북 train_food_merge_colab.ipynb 가 부른다.
#
# Food-101 에는 박스가 없다. 두 모델로 가짜 라벨(pseudo label)을 만든다.
#  1) 위치 모델 YOLOE-11S("food·bowl·plate…" 고정 어휘)로 가장 큰 음식 자리 = 그 사진의 음식(새 클래스).
#  2) 기존 342/400종 모델(best.pt)이 0.5 이상으로 잡은 **다른** 한식 자리(감자튀김·샐러드 등)도 라벨로 넣는다 —
#     빼면 햄버거 옆 감자튀김이 "배경" 으로 학습돼 기존 감자튀김 클래스를 깎는다.
# 로컬에서 작게 돌려 검증한다: python merge_food101.py --dry  (학습은 하지 않는다)
import argparse, json, os, random, shutil
from pathlib import Path

# (Food-101 폴더명, 앱 이름). AI Hub 에 이미 있는 음식(비빔밥·초밥·회·만두·볶음밥·감자튀김·미소된장국·쌀국수·카레·스파게티·치킨윙)은 뺐다 —
# 같은 음식을 두 이름으로 가르치면 모델이 헷갈린다. 이름은 food_labels_400.txt 와 겹치지 않는 것을 확인했다.
NEW_CLASSES = [
    ('pizza', '피자'), ('hamburger', '햄버거'), ('hot_dog', '핫도그'), ('ramen', '라멘'), ('takoyaki', '타코야키'),
    ('steak', '스테이크'), ('tacos', '타코'), ('nachos', '나초'), ('caesar_salad', '시저샐러드'), ('club_sandwich', '클럽샌드위치'),
    ('grilled_cheese_sandwich', '그릴드치즈샌드위치'), ('french_toast', '프렌치토스트'), ('pancakes', '팬케이크'), ('waffles', '와플'),
    ('donuts', '도넛'), ('ice_cream', '아이스크림'), ('cheesecake', '치즈케이크'), ('chocolate_cake', '초콜릿케이크'), ('cup_cakes', '컵케이크'),
    ('macarons', '마카롱'), ('tiramisu', '티라미수'), ('churros', '추로스'), ('onion_rings', '어니언링'), ('garlic_bread', '마늘빵'),
    ('lasagna', '라자냐'), ('risotto', '리조또'), ('macaroni_and_cheese', '맥앤치즈'), ('fish_and_chips', '피시앤칩스'), ('pad_thai', '팟타이'),
    ('eggs_benedict', '에그베네딕트'), ('baby_back_ribs', '폭립'), ('chicken_quesadilla', '퀘사디아'), ('omelette', '오믈렛'),
    ('edamame', '에다마메'), ('spring_rolls', '스프링롤'),
]
PROMPTS = ['food', 'bowl of food', 'plate of food', 'dish', 'bowl', 'plate', 'rice', 'soup', 'side dish']


def read_names(dataset: Path):
    return [l.strip() for l in (dataset / 'food_labels.txt').read_text(encoding='utf-8').splitlines() if l.strip()]


def write_yaml(dataset: Path, names):
    lines = [f'path: {dataset.as_posix()}', 'train: images/train', 'val: images/val', 'names:']
    lines += [f'  {i}: {n}' for i, n in enumerate(names)]
    (dataset / 'data.yaml').write_text('\n'.join(lines) + '\n', encoding='utf-8')
    (dataset / 'food_labels.txt').write_text('\n'.join(names) + '\n', encoding='utf-8')


def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    ar = lambda r: (r[2] - r[0]) * (r[3] - r[1])
    return ix / (ar(a) + ar(b) - ix + 1e-9)


def main_box(yoloe, img_path):
    """위치 모델이 찾은 음식 자리 중 가장 큰 것(점수 0.1 이상). Food-101 사진은 음식 하나를 중심에 찍은 것이라 가장 큰 자리가 그 음식이다.
    못 찾으면 사진 가운데 90% 를 쓴다. 가로·세로는 모델이 실제로 읽은 크기(orig_shape)를 쓴다 — PIL 은 EXIF 회전을 무시해
    가로·세로가 뒤바뀐 사진에서 좌표가 1 을 넘었다(2026-09-28, 맥앤치즈 1장)."""
    r = yoloe.predict(str(img_path), conf=0.1, verbose=False, imgsz=640)[0]
    h, w = r.orig_shape
    boxes = [b for b in r.boxes.xyxy.tolist() if (b[2] - b[0]) * (b[3] - b[1]) < 0.98 * w * h]
    if not boxes:
        return [0.05 * w, 0.05 * h, 0.95 * w, 0.95 * h], False, w, h
    return max(boxes, key=lambda b: (b[2] - b[0]) * (b[3] - b[1])), True, w, h


def known_boxes(base, img_path, name_to_index, conf=0.5):
    """기존 모델이 확신하는 한식 자리들 → (새 인덱스, 박스)."""
    if base is None:
        return []
    r = base.predict(str(img_path), conf=conf, verbose=False, imgsz=640)[0]
    out = []
    for c, b in zip(r.boxes.cls.tolist(), r.boxes.xyxy.tolist()):
        name = base.names[int(c)]
        if name in name_to_index:
            out.append((name_to_index[name], b))
    return out


def yolo_line(ci, b, w, h):
    x0, y0, x1, y1 = max(0, b[0]), max(0, b[1]), min(w, b[2]), min(h, b[3])
    return f'{ci} {((x0 + x1) / 2 / w):.6f} {((y0 + y1) / 2 / h):.6f} {((x1 - x0) / w):.6f} {((y1 - y0) / h):.6f}'


def merge(dataset: Path, food101: Path, base_pt: str | None, per_train: int, per_val: int, seed=42, log_every=500):
    from ultralytics import YOLO, YOLOE
    random.seed(seed)
    old = read_names(dataset)
    names = old + [ko for _, ko in NEW_CLASSES]
    assert len(set(names)) == len(names), '새 이름이 기존 이름과 겹친다'
    name_to_index = {n: i for i, n in enumerate(names)}
    yoloe = YOLOE('yoloe-11s-seg.pt'); yoloe.set_classes(PROMPTS, yoloe.get_text_pe(PROMPTS))
    base = YOLO(base_pt) if base_pt and Path(base_pt).exists() else None
    print(f'기존 {len(old)}종 + 새 {len(NEW_CLASSES)}종 = {len(names)}종 · 기존 모델로 한식 자리 라벨: {"있음" if base else "없음(위치 모델만)"}', flush=True)

    split = {s: [l.strip() for l in (food101 / 'meta' / f'{s}.txt').read_text().splitlines() if l.strip()] for s in ('train', 'test')}
    stats = {'images': 0, 'no_region': 0, 'extra_known': 0}
    for k, (folder, ko) in enumerate(NEW_CLASSES):
        ci = name_to_index[ko]
        for s, n, sub in (('train', per_train, 'train'), ('test', per_val, 'val')):
            files = [f for f in split[s] if f.startswith(folder + '/')]
            random.shuffle(files)
            for f in files[:n]:
                src = food101 / 'images' / (f + '.jpg')
                dst_img = dataset / 'images' / sub / f'f101_{folder}_{Path(f).name}.jpg'
                dst_lbl = dataset / 'labels' / sub / f'f101_{folder}_{Path(f).name}.txt'
                mb, found, w, h = main_box(yoloe, src)
                lines = [yolo_line(ci, mb, w, h)]
                for kci, kb in known_boxes(base, src, name_to_index):
                    if iou(kb, mb) < 0.4:   # 주 음식 자리와 겹치면 주 음식(새 클래스)이 맞다고 본다
                        lines.append(yolo_line(kci, kb, w, h)); stats['extra_known'] += 1
                shutil.copyfile(src, dst_img)
                dst_lbl.write_text('\n'.join(lines) + '\n', encoding='utf-8')
                stats['images'] += 1; stats['no_region'] += (not found)
                if stats['images'] % log_every == 0:
                    print(f'  {stats["images"]}장 ({k + 1}/{len(NEW_CLASSES)} {ko})', flush=True)
    write_yaml(dataset, names)
    print(f'완료: Food-101 {stats["images"]}장 · 위치 모델이 자리를 못 찾아 가운데 90% 를 쓴 사진 {stats["no_region"]}장 · '
          f'함께 넣은 한식 자리 {stats["extra_known"]}개', flush=True)
    (dataset / 'merge_stats.json').write_text(json.dumps(stats, ensure_ascii=False), encoding='utf-8')
    return names


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--dry', action='store_true', help='로컬 검증: 작은 가짜 데이터셋에 클래스당 2장씩만')
    ap.add_argument('--dataset'); ap.add_argument('--food101'); ap.add_argument('--base')
    ap.add_argument('--per-train', type=int, default=500); ap.add_argument('--per-val', type=int, default=100)
    a = ap.parse_args()
    if a.dry:
        import tempfile
        tmp = Path(tempfile.mkdtemp())
        for s in ('train', 'val'):
            (tmp / 'images' / s).mkdir(parents=True); (tmp / 'labels' / s).mkdir(parents=True)
        src = Path(a.dataset)
        shutil.copyfile(src / 'food_labels.txt', tmp / 'food_labels.txt')
        names = merge(tmp, Path(a.food101), a.base, per_train=2, per_val=1, log_every=10)
        print('dry 결과 폴더:', tmp, '· 클래스', len(names))
        for p in sorted((tmp / 'labels' / 'train').glob('*.txt'))[:6]:
            print(' ', p.name, '|', p.read_text(encoding='utf-8').strip().replace('\n', ' / '))
        print((tmp / 'data.yaml').read_text(encoding='utf-8').splitlines()[:4], '…', (tmp / 'data.yaml').read_text(encoding='utf-8').splitlines()[-2:])
    else:
        merge(Path(a.dataset), Path(a.food101), a.base, a.per_train, a.per_val)


In [ ]:
# 5. 머지 — 새 35종 × (학습 500 + 검증 100)장에 가짜 라벨을 달아 AI Hub 데이터셋에 합친다
import sys; sys.path.insert(0, '/content')
import importlib, merge_food101; importlib.reload(merge_food101)
if not (DATASET / 'merge_stats.json').exists():
    NAMES = merge_food101.merge(DATASET, F101, BASE_PT, per_train=500, per_val=100)
else:
    NAMES = [l.strip() for l in (DATASET / 'food_labels.txt').read_text(encoding='utf-8').splitlines() if l.strip()]
    print('이미 머지됨 —', len(NAMES), '종')
DATA_YAML = DATASET / 'data.yaml'


In [ ]:
# 5-2. UEC FOOD-256 받기(Kaggle 사본 rkuo2000/uecfood256, 로그인 없이 받아진다) + 머지
!pip install -q kagglehub
import kagglehub
UEC_ROOT = Path(kagglehub.dataset_download('rkuo2000/uecfood256'))
UEC_DIR = next(p for p in [UEC_ROOT / 'UECFOOD256', UEC_ROOT] if (p / '1' / 'bb_info.txt').exists())
print('UEC:', UEC_DIR)


In [ ]:
%%writefile /content/merge_uec.py
# UEC FOOD-256(일본 전기통신대, 비상업 연구용)을 머지 데이터셋에 더한다. Food-101 과 달리 **진짜 박스**가 있고
# 한 사진에 여러 음식이 담긴 식탁 사진이 섞여 있다 — 우리 모델의 약점("여러 음식 장면", FOOD_EVAL §3)을 직접 보강한다.
#
# 음식마다 셋 중 하나로 정한다(MAP):
#  - 기존 이름(AI Hub 342종·Food-101 35종)에 붙인다 — 같은 음식을 두 이름으로 가르치지 않는다.
#  - 새 이름으로 더한다(NEW) — 규동·가츠동·텐동처럼 대학생이 자주 먹는데 아직 없는 음식.
#  - 표에 없는 음식이 **한 번이라도** 박스로 들어 있는 사진은 통째로 뺀다 — 라벨 없는 음식이 사진에 남으면 모델이 그걸 "배경" 으로 배운다.
# UEC 폴더 구조: <카테고리 번호>/<사진 id>.jpg + bb_info.txt("img x1 y1 x2 y2"). 여러 음식 사진은 카테고리 폴더마다 같은 id 로 들어 있다.
# 로컬 검증: python merge_uec.py --dry --uec <UECFOOD256 폴더> --labels <food_labels.txt>
import argparse, hashlib, json, shutil
from pathlib import Path

# UEC 번호 → 이름. 기존 이름에 붙이는 것.
MAP = {
    1: '쌀밥', 2: '장어덮밥', 134: '장어덮밥', 6: '카레라이스', 9: '볶음밥', 11: '일반비빔밥', 20: '일식우동', 21: '일식우동',
    22: '메밀국수', 36: '미소된장국', 42: '군만두', 81: '고기만두', 51: '탕수육', 53: '달걀찜', 60: '햄버거스테이크', 64: '마파두부',
    71: '달걀말이', 82: '오므라이스', 84: '토마토소스스파게티', 85: '새우튀김', 35: '채소튀김', 55: '닭튀김', 94: '주먹밥',
    106: '닭갈비', 115: '짬뽕', 139: '돈가스', 140: '돈가스', 174: '돈가스', 141: '치킨가스', 197: '치킨윙', 98: '감자튀김',
    206: '쌀국수', 243: '자장면',
    # Food-101 에서 이미 더한 것
    17: '햄버거', 18: '피자', 23: '라멘', 29: '타코야키', 61: '스테이크', 40: '오믈렛', 67: '오믈렛', 97: '핫도그', 114: '팬케이크',
    117: '티라미수', 118: '와플', 119: '치즈케이크', 123: '리조또', 126: '프렌치토스트', 148: '타코', 149: '나초', 153: '라자냐',
    154: '시저샐러드', 161: '도넛', 209: '스프링롤', 225: '추로스',
}
# UEC 번호 → 새 이름. 대학생이 자주 먹는 일식·빵·디저트 중 아직 없는 것.
NEW = {
    92: '규동', 5: '가츠동', 4: '오야코동', 10: '텐동', 76: '카이센동', 77: '카이센동', 83: '카츠카레', 28: '오코노미야키',
    96: '츠케멘', 26: '야키소바', 65: '야키토리', 32: '고로케', 86: '감자샐러드', 88: '마카로니샐러드', 87: '샐러드',
    68: '달걀프라이', 46: '연어구이', 13: '크루아상', 145: '베이글', 158: '머핀', 146: '스콘', 12: '토스트', 116: '크레페',
    120: '생크림케이크', 224: '브라우니', 160: '슈크림', 162: '애플파이', 163: '파르페', 130: '치킨너겟', 170: '샤오롱바오',
    220: '스팸무스비',
}
VAL_PERCENT = 15


def read_boxes(uec: Path):
    """사진 id → [(UEC 번호, x1, y1, x2, y2, 그 사진 경로)]."""
    per = {}
    for d in sorted(p for p in uec.iterdir() if p.is_dir() and p.name.isdigit()):
        cat = int(d.name)
        info = d / 'bb_info.txt'
        if not info.exists():
            continue
        for line in info.read_text(encoding='utf-8', errors='ignore').splitlines()[1:]:
            parts = line.split()
            if len(parts) != 5:
                continue
            img = d / f'{parts[0]}.jpg'
            if img.exists():
                per.setdefault(parts[0], []).append((cat, *map(float, parts[1:]), img))
    return per


def plan(uec: Path, old_names):
    """어떤 사진을 넣고 뺄지. 반환: (새 이름 목록, [(id, 사진, [(이름, 박스)])], 통계)."""
    new_names = []
    for n in NEW.values():
        if n not in new_names and n not in old_names:
            new_names.append(n)
    clash = [n for n in dict.fromkeys(NEW.values()) if n in old_names]
    assert not clash, f'새 이름이 기존 이름과 겹친다: {clash}'
    missing = sorted({n for n in MAP.values() if n not in old_names})
    assert not missing, f'붙일 기존 이름이 라벨에 없다: {missing}'
    names = list(old_names) + new_names
    per = read_boxes(uec)
    keep, dropped = [], 0
    for pid, boxes in per.items():
        if any(b[0] not in MAP and b[0] not in NEW for b in boxes):
            dropped += 1
            continue
        labeled = [((MAP.get(b[0]) or NEW[b[0]]), b[1:5]) for b in boxes]
        keep.append((pid, boxes[0][5], labeled))
    stats = {'사진': len(per), '넣음': len(keep), '뺌(표에 없는 음식 포함)': dropped,
             '여러 음식 사진(넣은 것)': sum(1 for _, _, l in keep if len(l) > 1), '새 이름': len(new_names)}
    return names, keep, stats


def merge(dataset: Path, uec: Path, dry=False):
    from PIL import Image, ImageOps
    old = [l.strip() for l in (dataset / 'food_labels.txt').read_text(encoding='utf-8').splitlines() if l.strip()]
    names, keep, stats = plan(uec, old)
    index = {n: i for i, n in enumerate(names)}
    per_class = {}
    for pid, img, labeled in keep:
        split = 'val' if int(hashlib.md5(pid.encode()).hexdigest(), 16) % 100 < VAL_PERCENT else 'train'
        with Image.open(img) as im:
            w, h = ImageOps.exif_transpose(im).size if im.getexif().get(0x0112, 1) != 1 else im.size
        lines = []
        for n, (x1, y1, x2, y2) in labeled:
            x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
            if x2 - x1 < 2 or y2 - y1 < 2:
                continue
            lines.append(f'{index[n]} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}')
            per_class[n] = per_class.get(n, 0) + 1
        if not lines or dry:
            continue
        shutil.copyfile(img, dataset / 'images' / split / f'uec_{pid}.jpg')
        (dataset / 'labels' / split / f'uec_{pid}.txt').write_text('\n'.join(lines) + '\n', encoding='utf-8')
    stats['박스(이름별)'] = dict(sorted(per_class.items(), key=lambda x: -x[1]))
    if not dry:
        lines = [f'path: {dataset.as_posix()}', 'train: images/train', 'val: images/val', 'names:'] + [f'  {i}: {n}' for i, n in enumerate(names)]
        (dataset / 'data.yaml').write_text('\n'.join(lines) + '\n', encoding='utf-8')
        (dataset / 'food_labels.txt').write_text('\n'.join(names) + '\n', encoding='utf-8')
        (dataset / 'uec_stats.json').write_text(json.dumps(stats, ensure_ascii=False, indent=1), encoding='utf-8')
    print(json.dumps({k: v for k, v in stats.items() if k != '박스(이름별)'}, ensure_ascii=False), flush=True)
    return names, stats


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--uec', required=True); ap.add_argument('--labels', required=True); ap.add_argument('--dry', action='store_true')
    a = ap.parse_args()
    import tempfile
    tmp = Path(tempfile.mkdtemp()); shutil.copyfile(a.labels, tmp / 'food_labels.txt')
    names, stats = merge(tmp, Path(a.uec), dry=True)
    print('전체', len(names), '종 · 새', stats['새 이름'])
    print(json.dumps(stats['박스(이름별)'], ensure_ascii=False))


In [ ]:
import merge_uec; importlib.reload(merge_uec)
if not (DATASET / 'uec_stats.json').exists():
    NAMES, uec_stats = merge_uec.merge(DATASET, UEC_DIR)
else:
    NAMES = [l.strip() for l in (DATASET / 'food_labels.txt').read_text(encoding='utf-8').splitlines() if l.strip()]
    print('이미 머지됨 —', len(NAMES), '종')


In [ ]:
# 5-3. AI Hub 242 웹 음식 사진(PC 에서 aihub242_fetch.py 로 모아 Drive 에 올린 aihub242_web.tar) 머지
# 한식·양식·일식이 같은 웹 사진 스타일이라 "실사 = 서양 음식" 지름길을 끊는다(FOOD_EVAL §10).
WEB_TAR = TREX / 'aihub242_web.tar'
assert WEB_TAR.exists(), f'{WEB_TAR} 이 없다 — PC 에서 만든 tar 를 Drive trex/ 에 올린다'
if not Path('/content/web242/labels').exists():
    !mkdir -p /content/web242 && tar -xf "{WEB_TAR}" -C /content/web242
WEB_DIR = next(p.parent for p in Path('/content/web242').rglob('meta.tsv'))
print('웹 사진:', len(list((WEB_DIR / 'images').glob('*.jpg'))), '장')


In [ ]:
%%writefile /content/merge_web242.py
# PC 에서 aihub242_fetch.py 로 모은 웹 음식 사진(images/w242_*.jpg + labels/*.txt, 라벨의 첫 칸은 음식 이름)을
# 머지 데이터셋에 더한다. 번호는 여기서 매긴다 — 이미 있는 이름은 그 번호, 새 이름(배추김치 등)은 뒤에 붙인다.
# 이름당 최대 PER_CLASS 장(여러 묶음·여러 242 이름이 한 이름으로 모이면 합쳐서), 15% 는 검증으로.
import hashlib, json, random, shutil
from pathlib import Path

PER_CLASS = 400
VAL_PERCENT = 15


def merge(dataset: Path, web: Path, seed=42):
    names = [l.strip() for l in (dataset / 'food_labels.txt').read_text(encoding='utf-8').splitlines() if l.strip()]
    per = {}
    for lbl in sorted((web / 'labels').glob('*.txt')):
        lines = [l.split('\t') for l in lbl.read_text(encoding='utf-8').splitlines() if '\t' in l]
        if lines:
            per.setdefault(lines[0][0], []).append((lbl.stem, lines))
    rng = random.Random(seed)
    added = {}
    for name in sorted(per):
        if name not in names:
            names.append(name)
        idx = names.index(name)
        items = per[name]; rng.shuffle(items)
        for stem, lines in items[:PER_CLASS]:
            split = 'val' if int(hashlib.md5(stem.encode()).hexdigest(), 16) % 100 < VAL_PERCENT else 'train'
            shutil.copyfile(web / 'images' / f'{stem}.jpg', dataset / 'images' / split / f'{stem}.jpg')
            (dataset / 'labels' / split / f'{stem}.txt').write_text(
                '\n'.join(f'{names.index(n)} {c}' for n, c in lines) + '\n', encoding='utf-8')
            added[name] = added.get(name, 0) + 1
    lines = [f'path: {dataset.as_posix()}', 'train: images/train', 'val: images/val', 'names:'] + [f'  {i}: {n}' for i, n in enumerate(names)]
    (dataset / 'data.yaml').write_text('\n'.join(lines) + '\n', encoding='utf-8')
    (dataset / 'food_labels.txt').write_text('\n'.join(names) + '\n', encoding='utf-8')
    stats = {'이름': len(added), '사진': sum(added.values()), '전체 종': len(names), '이름별': added}
    (dataset / 'web242_stats.json').write_text(json.dumps(stats, ensure_ascii=False, indent=1), encoding='utf-8')
    print(json.dumps({k: v for k, v in stats.items() if k != '이름별'}, ensure_ascii=False), flush=True)
    return names


In [ ]:
import merge_web242; importlib.reload(merge_web242)
if not (DATASET / 'web242_stats.json').exists():
    NAMES = merge_web242.merge(DATASET, WEB_DIR)
else:
    NAMES = [l.strip() for l in (DATASET / 'food_labels.txt').read_text(encoding='utf-8').splitlines() if l.strip()]
    print('이미 머지됨 —', len(NAMES), '종')


In [ ]:
# 6. 학습 — 이전 모델이 있으면 그 가중치에서 시작한다(머리만 새 종 수로 다시 맞춘다). 끊기면 RESUME = True 로 다시 실행
from ultralytics import YOLO
RESUME = False
last = Path(RUNS_DIR) / 'merge' / 'weights' / 'last.pt'
if RESUME and last.exists():
    YOLO(str(last)).train(resume=True)
else:
    YOLO(BASE_PT or 'yolov8n.pt').train(
        data=str(DATA_YAML), epochs=80, patience=15, imgsz=640, batch=64, workers=8,
        cache='ram',   # 메모리가 모자라면 ultralytics 가 알아서 끈다
        cos_lr=True, seed=42, project=RUNS_DIR, name='merge', exist_ok=True,
    )
BEST = str(Path(RUNS_DIR) / 'merge' / 'weights' / 'best.pt')
print('best:', BEST)


In [ ]:
# 7. TFLite INT8 변환 + 앱 호환 검사 + Drive 저장 (평가보다 먼저 — 평가가 실패해도 모델은 남는다)
import tensorflow as tf, json, numpy as np
YOLO(BEST).export(format='tflite', int8=True, imgsz=640, data=str(DATA_YAML))
cands = sorted(Path(BEST).parent.rglob('*int8*.tflite'))
assert cands, 'INT8 tflite 를 찾지 못했다'
TFLITE = cands[0]
it = tf.lite.Interpreter(model_path=str(TFLITE)); it.allocate_tensors()
i, o = it.get_input_details()[0], it.get_output_details()[0]
assert i['dtype'] == np.float32 and o['dtype'] == np.float32, (i['dtype'], o['dtype'])
assert len(NAMES) + 4 in list(o['shape']), f"출력 {o['shape']} 가 {len(NAMES)}종과 안 맞는다"
shutil.copy(TFLITE, OUT_DIR / 'yolov8n_food.tflite')
(OUT_DIR / 'food_labels.txt').write_text('\n'.join(NAMES) + '\n', encoding='utf-8')
if (DATASET / 'nutrition.json').exists(): shutil.copy(DATASET / 'nutrition.json', OUT_DIR / 'nutrition_aihub.json')
(OUT_DIR / 'new_classes.json').write_text(json.dumps([n for n in NAMES if n not in OLD_NAMES], ensure_ascii=False), encoding='utf-8')
shutil.copy(DATASET / 'merge_stats.json', OUT_DIR / 'merge_stats.json')
if (DATASET / 'uec_stats.json').exists(): shutil.copy(DATASET / 'uec_stats.json', OUT_DIR / 'uec_stats.json')
print('완료 — Drive MyDrive/trex/merge_out/ 에 저장:', sorted(os.listdir(OUT_DIR)))
print(f"입력 {list(i['shape'])} · 출력 {list(o['shape'])} · {len(NAMES)}종")


In [ ]:
# 8. 망각 확인 — 같은 AI Hub 검증 사진으로 이전 모델 vs 새 모델, 그리고 새 35종 (실패해도 노트북을 멈추지 않는다)
import numpy as np, json, traceback
def val_on(model_path, names, images, tag):
    lst = Path(f'/content/{tag}.txt'); lst.write_text('\n'.join(images) + '\n')
    y = Path(f'/content/{tag}.yaml')
    y.write_text(f'path: {DATASET}\ntrain: {lst}\nval: {lst}\nnames:\n' + ''.join(f'  {i}: {n}\n' for i, n in enumerate(names)), encoding='utf-8')
    m = YOLO(model_path).val(data=str(y), imgsz=640, batch=64, verbose=False, plots=False)
    return dict(zip([names[i] for i in m.box.ap_class_index], m.box.ap50.tolist())), float(m.box.map50)

try:
    summary = {}
    new_old, new_old_map = val_on(BEST, NAMES, AIHUB_VAL, 'val_aihub_new')
    summary['새 모델 · AI Hub 검증 mAP50'] = round(new_old_map, 3)
    if BASE_PT:
        base_names = list(YOLO(BASE_PT).names.values())
        base_imgs = AIHUB_VAL
        old_old, old_old_map = val_on(BASE_PT, base_names, base_imgs, 'val_aihub_old')
        common = [n for n in old_old if n in new_old]
        summary['이전 모델 · AI Hub 검증 mAP50'] = round(old_old_map, 3)
        summary['공통 클래스 평균 AP50 (이전 → 새)'] = f"{np.mean([old_old[n] for n in common]):.3f} → {np.mean([new_old[n] for n in common]):.3f} ({len(common)}종)"
        drops = sorted(((new_old[n] - old_old[n], n) for n in common))[:10]
        summary['가장 많이 떨어진 10종'] = [f'{n} {d:+.2f}' for d, n in drops]
    f101_val = sorted(str(p) for p in (DATASET / 'images' / 'val').glob('f101_*.jpg'))
    new_new, new_new_map = val_on(BEST, NAMES, f101_val, 'val_f101')
    summary['새 35종 · Food-101 검증 mAP50'] = round(new_new_map, 3)
    summary['새 35종 클래스별 AP50'] = {n: round(v, 2) for n, v in sorted(new_new.items(), key=lambda x: -x[1]) if n not in OLD_NAMES}
    uec_val = sorted(str(p) for p in (DATASET / 'images' / 'val').glob('uec_*.jpg'))
    if uec_val:   # UEC 판: 실제 식탁(여러 음식) 사진이 섞인 검증 — 기존·새 음식 모두
        uec_ap, uec_map = val_on(BEST, NAMES, uec_val, 'val_uec')
        summary['UEC 검증(식탁 사진 포함) mAP50'] = round(uec_map, 3)
        summary['UEC 검증 클래스별 AP50'] = {n: round(v, 2) for n, v in sorted(uec_ap.items(), key=lambda x: -x[1])}
    print(json.dumps(summary, ensure_ascii=False, indent=1))
    (OUT_DIR / 'eval_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=1), encoding='utf-8')
except Exception:
    traceback.print_exc(); print('평가 실패 — 모델은 7번 셀에서 이미 Drive 에 저장됐다')


In [ ]:
%%writefile /content/compare_models.py
# 음식 모델 두 개를 식탁 사진 16장에서 앱과 같은 방식으로 견준다 — 전체 사진 1회 / 2단계(위치 모델 → 회색 여백 30% 자르기 → 이름).
# TensorFlow 없이 LiteRT(ai_edge_litert)만 쓴다(이 PC 는 보안 정책이 TF DLL 을 막는다, 2026-09-28).
# 실행: python compare_models.py <옛 모델.tflite> <옛 라벨.txt> <새 모델.tflite> <새 라벨.txt>
import csv, sys
from pathlib import Path
import numpy as np
from PIL import Image
from ai_edge_litert.interpreter import Interpreter

REPO = Path(__file__).resolve().parents[2]
import os
ROOT = Path(os.environ.get('TREX_EVAL_REAL', REPO.parent / 'aihub74_raw' / 'eval_real'))
REGION = Path(os.environ.get('TREX_REGION_MODEL', REPO / 'app/src/main/assets/models/food_region.tflite'))
THRESHOLD = 0.40
# 새 음식이 생기면 "맞힐 수 있는 음식" 도 늘어난다 — 원래 정답표(342종 기준)에 더한다.
EXTRA_IN_CLASS = {'photo05.jpg': ['피자'], 'photo14.jpg': ['고로케'], 'photo02.jpg': ['야키토리']}   # 라벨에 없는 모델에서는 자동으로 빠진다


class Model:
    def __init__(self, path):
        self.it = Interpreter(model_path=str(path), num_threads=4); self.it.allocate_tensors()
        self.i, self.o = self.it.get_input_details()[0], self.it.get_output_details()[0]
        sh = self.i['shape']; self.cf = sh[1] == 3 and sh[3] != 3; self.S = sh[2] if self.cf else sh[1]

    def run(self, canvas):
        a = np.asarray(canvas, np.float32)[None] / 255.0
        if self.cf: a = a.transpose(0, 3, 1, 2)
        self.it.set_tensor(self.i['index'], a); self.it.invoke()
        o = self.it.get_tensor(self.o['index'])[0]
        return o if o.shape[0] < o.shape[1] else o.T

    def letterbox(self, im):
        S = self.S; s = min(S / im.width, S / im.height); nw, nh = max(1, int(im.width * s)), max(1, int(im.height * s))
        c = Image.new('RGB', (S, S), (114, 114, 114)); c.paste(im.resize((nw, nh), Image.BILINEAR), ((S - nw) // 2, (S - nh) // 2))
        return c, s, (S - nw) // 2, (S - nh) // 2


def scores(model, canvas):
    return model.run(canvas)[4:].max(1)


def gray30(im, b, S):
    c = im.crop(tuple(int(v) for v in b)); side = int(max(c.width, c.height) / np.sqrt(0.3))
    g = Image.new('RGB', (side, side), (114, 114, 114)); g.paste(c, ((side - c.width) // 2, (side - c.height) // 2))
    return g.resize((S, S), Image.BILINEAR)


def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    ar = lambda r: (r[2] - r[0]) * (r[3] - r[1]); return ix / (ar(a) + ar(b) - ix + 1e-9)


def regions(rm, im):
    """앱 FoodRegions.decode + tidy 와 같은 규칙."""
    c, s, px, py = rm.letterbox(im); o = rm.run(c)
    best = o[4:].max(0); sel = np.where(best >= 0.05)[0]
    cx, cy, w, h = o[0, sel], o[1, sel], o[2, sel], o[3, sel]
    b = np.stack([(cx - w/2 - px) / s, (cy - h/2 - py) / s, (cx + w/2 - px) / s, (cy + h/2 - py) / s], 1)
    b[:, [0, 2]] = b[:, [0, 2]].clip(0, im.width); b[:, [1, 3]] = b[:, [1, 3]].clip(0, im.height)
    order = best[sel].argsort()[::-1]; kept = []
    for k in order:
        if all(iou(b[k], b[j]) < 0.5 for j in kept): kept.append(k)
    boxes = [b[k].tolist() for k in kept if (b[k][2]-b[k][0])*(b[k][3]-b[k][1]) <= 0.6 * im.width * im.height]
    area = lambda r: (r[2]-r[0])*(r[3]-r[1])
    out = [r for r in boxes if not any(area(q) > area(r) and max(0, min(r[2], q[2]) - max(r[0], q[0])) * max(0, min(r[3], q[3]) - max(r[1], q[1])) / max(area(r), 1e-9) >= 0.85 for q in boxes)]
    return out[:20]


def evaluate(path, labels_path, rm, rows, imgs):
    m = Model(path); L = [l.strip() for l in open(labels_path, encoding='utf-8-sig') if l.strip()]
    res = {}
    for mode in ('전체 1회', '2단계'):
        hit = miss = extra = top3 = 0; want_total = 0
        for r in rows:
            im = imgs[r['file']]
            want = [x for x in r['expected_in_class'].split('|') if x] + EXTRA_IN_CLASS.get(r['file'], [])
            want = [w for w in dict.fromkeys(want) if w in L]; want_total += len(want)
            if mode == '전체 1회':
                s = scores(m, m.letterbox(im)[0]); got = [L[i] for i in np.argsort(-s) if s[i] >= THRESHOLD][:5]
                rank = {w: list(np.argsort(-s)).index(L.index(w)) for w in want}
            else:
                got, rank = [], {w: 999 for w in want}
                for b in regions(rm, im):
                    s = scores(m, gray30(im, b, m.S)); o = list(np.argsort(-s))
                    if s[o[0]] >= THRESHOLD: got.append(L[o[0]])
                    for w in want: rank[w] = min(rank[w], o.index(L.index(w)))
                got = list(dict.fromkeys(got))
            hit += len([g for g in got if g in want]); miss += len([w for w in want if w not in got]); extra += len([g for g in got if g not in want])
            top3 += sum(1 for v in rank.values() if v < 3)
        res[mode] = (hit, want_total, miss, extra, top3)
    return res


if __name__ == '__main__':
    old_m, old_l, new_m, new_l = sys.argv[1:5]
    rows = list(csv.DictReader(open(ROOT / 'ground_truth_multi.csv', encoding='utf-8')))
    imgs = {r['file']: Image.open(ROOT / 'images' / r['file']).convert('RGB') for r in rows}
    rm = Model(REGION)
    for tag, mp, lp in (('옛 모델', old_m, old_l), ('새 모델', new_m, new_l)):
        for mode, (h, wt, mi, ex, t3) in evaluate(mp, lp, rm, rows, imgs).items():
            print(f'{tag} · {mode}: 검출 {h}/{wt} ({h/max(wt,1):.0%}) · 누락 {mi} · 오검출 {ex} · 손볼/장 {(mi+ex)/len(rows):.2f} · 정답 후보 3위 안 {t3}/{wt}', flush=True)


In [ ]:
# 9. 식탁 사진 16장 판정 — Colab 검증 점수는 학습과 같은 분포라 실사용 성능을 못 본다(FOOD_EVAL §10). 앱과 같은 2단계로 잰다.
# 필요한 것: Drive trex/eval_real.tar(사진 16장 + 정답표). 현재 앱 모델(342)·위치 모델은 GitHub 에서 받는다.
!pip install -q ai-edge-litert
import zipfile, subprocess
REAL_TAR = TREX / 'eval_real.tar'
if REAL_TAR.exists():
    !mkdir -p /content/real && tar -xf "{REAL_TAR}" -C /content/real
    REAL = next(p.parent for p in Path('/content/real').rglob('ground_truth_multi.csv'))
    RAW = 'https://raw.githubusercontent.com/LeeDongHyun00/TREX/feature/food-two-stage/app/src/main/assets/models/'
    for f in ('food_region.tflite', 'yolov8n_food.tflite', 'food_labels.txt'):
        !curl -sL -o /content/app_{f} {RAW}{f}
    def labels_of(tfl, out):   # ultralytics tflite 는 이름 목록을 파일 안(metadata.json)에 담는다
        m = json.loads(zipfile.ZipFile(tfl).read('metadata.json')); names = [m['names'][k] for k in sorted(m['names'], key=int)]
        Path(out).write_text('\n'.join(names) + '\n', encoding='utf-8'); return out
    new_l = labels_of(str(OUT_DIR / 'yolov8n_food.tflite'), '/content/new_labels.txt')
    env = dict(os.environ, TREX_EVAL_REAL=str(REAL), TREX_REGION_MODEL='/content/app_food_region.tflite')
    r = subprocess.run(['python', '/content/compare_models.py', '/content/app_yolov8n_food.tflite', '/content/app_food_labels.txt',
                        str(OUT_DIR / 'yolov8n_food.tflite'), new_l], capture_output=True, text=True, env=env)
    print(r.stdout[-2000:], r.stderr[-1500:])
    (OUT_DIR / 'real16.txt').write_text(r.stdout, encoding='utf-8')
    print('판정: 새 모델의 "2단계" 검출이 옛 모델(342)의 13/25(52%) 보다 높고 오검출이 크게 늘지 않아야 앱에 넣는다.')
else:
    print('Drive trex/eval_real.tar 가 없어 식탁 사진 판정을 건너뛴다 — PC 에서 compare_models.py 로 잰다.')


### 아침에 할 일
- `MyDrive/trex/merge_out/eval_summary.json` 으로 **기존 음식 성능이 유지됐는지**(공통 클래스 AP50 이전 → 새)와 새 35종 성능을 본다.
- 괜찮으면 `yolov8n_food.tflite`·`food_labels.txt` 를 앱 `assets/models/` 에 넣고, 새 35종 영양값을 `TrexData.kt` 에 추가한다(Food-101 에는 영양값이 없다 — 추정치로 표시).
- 세션이 끊겼으면: 1~3번 셀 → 6번 셀에서 `RESUME = True` → 7·8번 셀. (4·5번은 이미 했으면 건너뛴다.)
